In [1]:
import os
from math import prod
from safetensors import safe_open
import plotly.graph_objects as go

# =====================================================================
# 1. 데이터 추출 함수 (순수 텐서 용량 기준)
# =====================================================================
def analyze_safetensors_gib(file_path):
    if not os.path.exists(file_path): return 0.0
    tensor_bytes_sum = 0
    with safe_open(file_path, framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor = f.get_tensor(key)
            params = prod(tensor.shape)
            element_size = tensor.element_size() 
            tensor_bytes_sum += (params * element_size)
    return tensor_bytes_sum / (1024 ** 3)

# =====================================================================
# 2. 경로 설정 및 데이터 처리
# =====================================================================
BASE_ORIGINAL = r"C:\Users\user\SLM\00_Base_Models"
BASE_QUANT = r"C:\Users\user\SLM\02_cuda_aligned"

models_config = {
    "Llama_3.2_1B": [
        rf"{BASE_ORIGINAL}\Llama-3.2-1B-Instruct\model.safetensors",
        rf"{BASE_QUANT}\Llama_3.2_1B\Distillation\GPTQ_8bit\model.safetensors",
        rf"{BASE_QUANT}\Llama_3.2_1B\Distillation\GPTQ_4bit\model.safetensors",
        rf"{BASE_QUANT}\Llama_3.2_1B\Distillation\GPTQ_3bit\model.safetensors",
        rf"{BASE_QUANT}\Llama_3.2_1B\Distillation\GPTQ_2bit\model.safetensors"
    ],
    "Qwen2.5_1.5B": [
        rf"{BASE_ORIGINAL}\Qwen2.5-1.5B-Instruct\model.safetensors",
        rf"{BASE_QUANT}\Qwen2.5_1.5B\Base_RLHF\GPTQ_8bit\model.safetensors",
        rf"{BASE_QUANT}\Qwen2.5_1.5B\Base_RLHF\GPTQ_4bit\model.safetensors",
        rf"{BASE_QUANT}\Qwen2.5_1.5B\Base_RLHF\GPTQ_3bit\model.safetensors",
        rf"{BASE_QUANT}\Qwen2.5_1.5B\Base_RLHF\GPTQ_2bit\model.safetensors"
    ],
    "TinyLlama_1.1B": [
        rf"{BASE_ORIGINAL}\TinyLlama-1.1B-Chat-v1.0\model.safetensors",
        rf"{BASE_QUANT}\TinyLlama_1.1B\Base_Scratch\GPTQ_8bit\model.safetensors",
        rf"{BASE_QUANT}\TinyLlama_1.1B\Base_Scratch\GPTQ_4bit\model.safetensors",
        rf"{BASE_QUANT}\TinyLlama_1.1B\Base_Scratch\GPTQ_3bit\model.safetensors",
        rf"{BASE_QUANT}\TinyLlama_1.1B\Base_Scratch\GPTQ_2bit\model.safetensors"
    ]
}

# =====================================================================
# 3. 단계별 한계 감소율(Marginal Reduction) 계산
# =====================================================================
marginal_data = {}
for model, paths in models_config.items():
    sizes = [analyze_safetensors_gib(p) for p in paths] # [Base, 8b, 4b, 3b, 2b]
    
    rates = []
    diffs = []
    
    for i in range(4): 
        prev_size = sizes[i]
        curr_size = sizes[i+1]
        
        if prev_size > 0 and curr_size > 0:
            reduction_rate = ((prev_size - curr_size) / prev_size) * 100
            diff_gib = prev_size - curr_size
            rates.append(reduction_rate)
            diffs.append(diff_gib)
        else:
            rates.append(0.0)
            diffs.append(0.0)
            
    marginal_data[model] = {"rates": rates, "diffs": diffs}

# =====================================================================
# 4. Plotly 시각화 
# =====================================================================
labels_transitions = ['원본 → 8-bit', '8-bit → 4-bit (가성비 스위트 스팟)', '4-bit → 3-bit', '3-bit → 2-bit']

# 막대 채우기 색상
colors = {
    'Llama_3.2_1B': '#BDE0FE',   
    'Qwen2.5_1.5B': '#E1C6E9',   
    'TinyLlama_1.1B': '#FFE0B2'  
}

# [추가] 막대 테두리용 진한 색상
line_colors = {
    'Llama_3.2_1B': '#90CAF9',   
    'Qwen2.5_1.5B': '#CE93D8',   
    'TinyLlama_1.1B': '#FFB74D'  
}

fig = go.Figure()

for model_name, data in marginal_data.items():
    rates = data["rates"]
    diffs = data["diffs"]
    
    text_labels = [
        f"<b>{r:.1f}%</b><br><span style='font-size:11px; color:#555;'>↓ {d:.2f} GiB</span>" if r > 0 else "" 
        for r, d in zip(rates, diffs)
    ]
    
    fig.add_trace(go.Bar(
        name=model_name,
        x=labels_transitions,
        y=rates,
        text=text_labels,
        textposition='outside',
        textfont=dict(size=15, color='#000000'),
        marker_color=colors[model_name],
        # [수정됨] 모델 이름에 매핑된 테두리 색상 적용
        marker_line_color=line_colors[model_name],
        marker_line_width=1.5,
        hovertemplate="<b>%{x}</b><br>모델: " + model_name + "<br>직전 단계 대비 추가 감소율: %{y:.2f}%<br>추가 확보 용량: %{customdata:.2f} GiB<extra></extra>",
        customdata=diffs 
    ))

max_rate = max([max(data["rates"]) for data in marginal_data.values()]) if marginal_data else 100

fig.update_layout(
    title=dict(text='<b>양자화 단계별 한계 압축 효율 (직전 정밀도 대비 추가 감소율)</b>', font=dict(size=22), x=0.5, y=0.92),
    yaxis_title=dict(text='<b>직전 단계 대비 용량 감소율 (%)</b>', font=dict(size=15)),
    barmode='group',
    bargroupgap=0.15,
    bargap=0.2,
    yaxis=dict(
        range=[0, max_rate + 12], # 상단 텍스트 공간 확보
        showgrid=True, gridcolor='#EAEAEA', zeroline=True, zerolinecolor='#CCCCCC'
    ),
    xaxis=dict(
        tickfont=dict(size=14, weight='bold'),
        # [핵심] 범주형 X축의 좌우(0번 인덱스 왼쪽, 3번 인덱스 오른쪽)에 0.7만큼의 빈 공간을 추가
        range=[-0.7, 3.7] 
    ),
    font=dict(family="Malgun Gothic, AppleGothic, NanumGothic, sans-serif"),
    plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=13)),
    margin=dict(t=120, b=50, l=50, r=50)
)

fig.show()

In [2]:
import os
import re
import plotly.graph_objects as go

# =====================================================================
# 1. 데이터 파싱 및 추출 함수
# =====================================================================
def parse_log_file(file_path):
    if not os.path.exists(file_path): return [0.0] * 5
    with open(file_path, 'r', encoding='utf-8') as f:
        log_text = f.read()
    try:
        base_vram = float(re.search(r'순수 모델 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_8bit = float(re.search(r'8-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_4bit = float(re.search(r'4-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_3bit = float(re.search(r'3-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_2bit = float(re.search(r'2-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
    except AttributeError:
        return [0.0] * 5
    return [base_vram, vram_8bit, vram_4bit, vram_3bit, vram_2bit]

def calculate_reduction_rates(data_list):
    base_val = data_list[0]
    if base_val == 0: return [0.0] * 5
    return [0.0] + [((base_val - val) / base_val) * 100 for val in data_list[1:]]

# =====================================================================
# 2. 경로 설정 및 데이터 연산
# =====================================================================
log_paths = {
    'Llama_3.2_1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Llama_3.2_1B\master_log_Llama_3.2_1B_Distillation_20260406_031434.log',
    'Qwen2.5_1.5B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Qwen2.5_1.5B\master_log_Qwen2.5_1.5B_Base_RLHF_20260406_010213.log',
    'TinyLlama_1.1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\TinyLlama_1.1B\master_log_TinyLlama_1.1B_Base_Scratch_20260405_231336.log'
}

model_rates = {}
for model_name, path in log_paths.items():
    vram_values = parse_log_file(path)
    model_rates[model_name] = calculate_reduction_rates(vram_values)

# X축 라벨 선언
labels = ['원본 (BF16)', 'GPTQ 8-bit', 'GPTQ 4-bit', 'GPTQ 3-bit', 'GPTQ 2-bit']

# 선 및 텍스트 색상 (진한 톤 유지)
line_colors = {'Llama_3.2_1B': '#5A9BD5', 'Qwen2.5_1.5B': '#9B59B6', 'TinyLlama_1.1B': '#E67E22'}

# =====================================================================
# 3. 단일 선형 그래프 시각화 (X축 라벨 포함)
# =====================================================================
fig = go.Figure()

for model_name, rates in model_rates.items():
    # 1. 선과 마커 (X축 라벨 적용)
    fig.add_trace(go.Scatter(
        name=model_name,
        x=labels, 
        y=rates,
        mode='lines+markers', 
        line=dict(color=line_colors[model_name], width=3),
        marker=dict(size=8, symbol='circle', color='white', line=dict(color=line_colors[model_name], width=2)),
        showlegend=True, # 단일 그래프이므로 범례 활성화
        hovertemplate="<b>%{x}</b><br>모델: " + model_name + "<br>감소율: %{y:.2f}%<extra></extra>"
    ))

    # 2. 텍스트 주석(Annotation) 개별 추가
    OFFSET_PX = 15 
    
    for i, rate in enumerate(rates):
        # 0% 중복 표기 방지 (기준점인 Qwen 하나만 남김)
        if i == 0 and model_name != 'Qwen2.5_1.5B':
            continue
            
        text_val = "<b>0.0%</b>" if i == 0 else f"<b>{rate:.1f}%</b>"
        text_col = 'black' if i == 0 else line_colors[model_name]
        
        # 텍스트 겹침 방지를 위한 상하 배치 제어
        is_top = True
        if model_name == 'Llama_3.2_1B' and i != 1: is_top = False
        if model_name == 'TinyLlama_1.1B' and i == 1: is_top = False
        
        shift_val = OFFSET_PX if is_top else -OFFSET_PX
        
        fig.add_annotation(
            x=labels[i],
            y=rate,
            text=text_val,
            font=dict(size=13, color=text_col),
            showarrow=False,
            yshift=shift_val
        )

# =====================================================================
# 4. 레이아웃 및 여백 설정 (단일 차트 맞춤형)
# =====================================================================
max_rate = max([max(r) for r in model_rates.values() if r]) if any(model_rates.values()) else 100

fig.update_layout(
    title=dict(
        text='<b>양자화 정밀도별 VRAM 감소율 추이 (%)</b>', 
        font=dict(size=22), x=0.5, y=0.92
    ),
    yaxis_title=dict(text='<b>VRAM 감소율 (%)</b>', font=dict(size=15)),
    height=550, # PPT 삽입에 적합하도록 높이 축소
    font=dict(family="Malgun Gothic, AppleGothic, NanumGothic, sans-serif"),
    plot_bgcolor='white',
    # 범례 중앙 정렬 배치
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5, 
        font=dict(size=13)
    ),
    margin=dict(t=100, b=60, l=60, r=40) # 하단 여백(b)을 충분히 주어 X축 라벨 공간 확보
)

fig.update_yaxes(range=[0, max_rate + 20], showgrid=True, gridcolor='#EAEAEA', zeroline=True, zerolinecolor='#CCCCCC')
fig.update_xaxes(tickfont=dict(size=13, weight='bold'))

fig.show()

In [3]:
import os
import re
import plotly.graph_objects as go

def parse_base_vram(file_path):
    if not os.path.exists(file_path):
        print(f"경고: {file_path} 파일을 찾을 수 없어 0.0을 반환합니다.")
        return 0.0

    with open(file_path, 'r', encoding='utf-8') as f:
        log_text = f.read()

    try:
        base_vram = float(re.search(r'순수 모델 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
    except AttributeError:
        print(f"오류: {file_path} 에서 VRAM 데이터를 파싱할 수 없습니다.")
        return 0.0

    return base_vram

# 1. 로그 파일 경로 지정
log_paths = {
    'Llama_3.2_1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Llama_3.2_1B\master_log_Llama_3.2_1B_Distillation_20260406_031434.log',
    'TinyLlama_1.1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\TinyLlama_1.1B\master_log_TinyLlama_1.1B_Base_Scratch_20260405_231336.log',
    'Qwen2.5_1.5B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Qwen2.5_1.5B\master_log_Qwen2.5_1.5B_Base_RLHF_20260406_010213.log'
}

# 2. 이론적 VRAM 적재량 (GiB 기준)
theoretical_vram = {
    'Llama_3.2_1B': 2.29,
    'TinyLlama_1.1B': 2.05,
    'Qwen2.5_1.5B': 2.87
}

# 3. 실제 로그 데이터 추출
actual_vram = {}
for model_name, path in log_paths.items():
    actual_vram[model_name] = parse_base_vram(path)

# =====================================================================
# 4. 그래프 시각화 (Plotly) - offsetgroup을 활용한 네이티브 매핑
# =====================================================================
models = ['Llama_3.2_1B', 'TinyLlama_1.1B', 'Qwen2.5_1.5B']
y_theoretical = [theoretical_vram[m] for m in models]

fig = go.Figure()

# 트레이스 1: 이론적 적재량 (공통 회색, offsetgroup='0')
fig.add_trace(go.Bar(
    name='이론적 VRAM (가중치 순수 용량)',
    x=models,
    y=y_theoretical,
    offsetgroup='0', # 첫 번째 기둥 슬롯에 고정
    text=[f"<b>{val:.2f}</b>" for val in y_theoretical], 
    textposition='outside',
    textfont=dict(size=14, color='#666666'),
    marker_color='#E0E0E0',
    marker_line_color='#BDBDBD',
    marker_line_width=1.5,
    hovertemplate="<b>%{x}</b><br>이론적 수치: %{y:.2f} GiB<extra></extra>"
))

# 트레이스 2: 실제 로그 적재량 - Llama (파란색, offsetgroup='1')
fig.add_trace(go.Bar(
    name='Llama 3.2 (실제 할당량)',
    x=models,
    y=[actual_vram['Llama_3.2_1B'], None, None], # Llama 자리에만 데이터 삽입
    offsetgroup='1', # 두 번째 기둥 슬롯 공유
    text=[f"<b>{actual_vram['Llama_3.2_1B']:.2f}</b>", "", ""],
    textposition='outside',
    textfont=dict(size=15, color='#333333'),
    marker_color='#BBDEFB',
    marker_line_color='#64B5F6',
    marker_line_width=1.5,
    hovertemplate="<b>%{x}</b><br>실제 적재량: %{y:.2f} GiB<extra></extra>"
))

# 트레이스 3: 실제 로그 적재량 - TinyLlama (주황색, offsetgroup='1')
fig.add_trace(go.Bar(
    name='TinyLlama (실제 할당량)',
    x=models,
    y=[None, actual_vram['TinyLlama_1.1B'], None], # TinyLlama 자리에만 데이터 삽입
    offsetgroup='1', # 두 번째 기둥 슬롯 공유
    text=["", f"<b>{actual_vram['TinyLlama_1.1B']:.2f}</b>", ""],
    textposition='outside',
    textfont=dict(size=15, color='#333333'),
    marker_color='#FFE0B2',
    marker_line_color='#FFB74D',
    marker_line_width=1.5,
    hovertemplate="<b>%{x}</b><br>실제 적재량: %{y:.2f} GiB<extra></extra>"
))

# 트레이스 4: 실제 로그 적재량 - Qwen (보라색, offsetgroup='1')
fig.add_trace(go.Bar(
    name='Qwen 2.5 (실제 할당량)',
    x=models,
    y=[None, None, actual_vram['Qwen2.5_1.5B']], # Qwen 자리에만 데이터 삽입
    offsetgroup='1', # 두 번째 기둥 슬롯 공유
    text=["", "", f"<b>{actual_vram['Qwen2.5_1.5B']:.2f}</b>"],
    textposition='outside',
    textfont=dict(size=15, color='#333333'),
    marker_color='#E1BEE7',
    marker_line_color='#BA68C8',
    marker_line_width=1.5,
    hovertemplate="<b>%{x}</b><br>실제 적재량: %{y:.2f} GiB<extra></extra>"
))

# Y축 상단 여백 계산
actual_vals = list(actual_vram.values())
max_val = max(max(y_theoretical), max(actual_vals) if any(actual_vals) else 0)

fig.update_layout(
    title=dict(
        text='<b>모델별 VRAM 적재량 비교: 이론적 수치 vs 실제 로그 분석</b>',
        font=dict(size=20), x=0.5, y=0.9
    ),
    yaxis_title=dict(text='<b>VRAM (GiB)</b>', font=dict(size=14)),
    barmode='group',
    bargroupgap=0.15,
    bargap=0.1,
    yaxis=dict(
        range=[0, max_val + 0.8],
        showgrid=True, gridcolor='#EAEAEA', zeroline=True, zerolinecolor='#CCCCCC'
    ),
    xaxis=dict(tickfont=dict(size=13, weight='bold')),
    font=dict(family="Malgun Gothic, AppleGothic, NanumGothic, sans-serif"),
    plot_bgcolor='white',
    # 범례 배치를 가운데 정렬하여 더욱 안정감 있게 조절
    legend=dict(
        orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5, 
        font=dict(size=12)
    ),
    margin=dict(t=120, b=50, l=50, r=50),
    uniformtext_minsize=14,
    uniformtext_mode='show'
)

fig.show()

In [4]:
import os
import re
import plotly.graph_objects as go

# =====================================================================
# 1. 데이터 파싱 및 추출 함수
# =====================================================================
def parse_log_file(file_path):
    if not os.path.exists(file_path): return [0.0] * 5
    with open(file_path, 'r', encoding='utf-8') as f:
        log_text = f.read()
    try:
        base_vram = float(re.search(r'순수 모델 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_8bit = float(re.search(r'8-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_4bit = float(re.search(r'4-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_3bit = float(re.search(r'3-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_2bit = float(re.search(r'2-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
    except AttributeError:
        return [0.0] * 5
    return [base_vram, vram_8bit, vram_4bit, vram_3bit, vram_2bit]

# =====================================================================
# 2. 경로 설정 및 절대 감소량(Marginal Savings) 계산
# =====================================================================
log_paths = {
    'Llama_3.2_1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Llama_3.2_1B\master_log_Llama_3.2_1B_Distillation_20260406_031434.log',
    'Qwen2.5_1.5B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Qwen2.5_1.5B\master_log_Qwen2.5_1.5B_Base_RLHF_20260406_010213.log',
    'TinyLlama_1.1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\TinyLlama_1.1B\master_log_TinyLlama_1.1B_Base_Scratch_20260405_231336.log'
}

marginal_savings = {}

for model_name, path in log_paths.items():
    vram_vals = parse_log_file(path)
    diffs = []
    for i in range(4):
        if vram_vals[i] > 0 and vram_vals[i+1] > 0:
            diffs.append(vram_vals[i] - vram_vals[i+1])
        else:
            diffs.append(0.0)
    marginal_savings[model_name] = diffs

# =====================================================================
# 3. Plotly 시각화 (프레임워크 오버헤드 수학적 복원 적용)
# =====================================================================
transition_labels = ['원본 → 8-bit', '8-bit → 4-bit', '4-bit → 3-bit', '3-bit → 2-bit']

colors = {'Llama_3.2_1B': '#BDE0FE', 'Qwen2.5_1.5B': '#E1C6E9', 'TinyLlama_1.1B': '#FFE0B2'}
line_colors = {'Llama_3.2_1B': '#90CAF9', 'Qwen2.5_1.5B': '#CE93D8', 'TinyLlama_1.1B': '#FFB74D'}

fig = go.Figure()

max_theoretical_height = 0

for model_name, diffs in marginal_savings.items():
    text_labels = [f"<b>↓ {val:.2f}</b>" if val > 0 else "" for val in diffs]
    
    # [1] 기본 막대 (실제 측정된 VRAM 확보량)
    fig.add_trace(go.Bar(
        name=model_name,
        x=transition_labels,
        y=diffs,
        offsetgroup=model_name, 
        text=text_labels,
        textposition='outside', 
        textfont=dict(size=14, color='#000000'), 
        marker_color=colors[model_name],
        marker_line_color=line_colors[model_name],
        marker_line_width=1.5,
        hovertemplate="<b>%{x}</b><br>모델: " + model_name + "<br>확보된 VRAM: %{y:.2f} GiB<extra></extra>"
    ))

    # --- 수학적 오버헤드 역산 로직 ---
    theoretical_savings_16_to_8 = diffs[1] * 2 
    hidden_tax = max(0, theoretical_savings_16_to_8 - diffs[0]) 

    if theoretical_savings_16_to_8 > max_theoretical_height:
        max_theoretical_height = theoretical_savings_16_to_8

    hidden_y = [hidden_tax, 0, 0, 0]
    base_y = [diffs[0], 0, 0, 0] 
    
    ghost_text = [
        f"<span style='font-size:12px; color:#D32F2F;'><b>+ {val:.2f}</b><br>가려진 오버헤드</span>" if val > 0 else "" 
        for val in hidden_y
    ]

    # [2] 유령 막대 (가려진 프레임워크 텍스 - Plotly 에러 수정본)
    fig.add_trace(go.Bar(
        name=f"{model_name} (숨겨진 공간)",
        x=transition_labels,
        y=hidden_y,
        base=base_y,            
        offsetgroup=model_name, 
        text=ghost_text,
        textposition='outside',
        # [핵심 수정] 점선(dash) 속성을 제거하고, 패턴(pattern) 딕셔너리를 사용하여 빗금 시각화
        marker=dict(
            color='rgba(255, 255, 255, 0)', # 막대 내부 투명도 유지
            line=dict(color='#D32F2F', width=1.5), # 실선 붉은 테두리
            pattern=dict(shape='/', fillmode='replace', fgcolor='#EF9A9A') # 옅은 붉은색 빗금 패턴 추가
        ),
        showlegend=False,
        hovertemplate=(
            "<b>%{x}</b><br>" +
            "가려진 엔진 오버헤드: %{y:.2f} GiB<br>" +
            "이론적 총 확보량: " + f"{theoretical_savings_16_to_8:.2f} GiB<extra></extra>"
        )
    ))

max_actual = max([max(diff) for diff in marginal_savings.values()]) if marginal_savings else 1.0
final_y_max = max(max_actual, max_theoretical_height)

fig.update_layout(
    title=dict(
        text='<b>양자화 단계별 순수 VRAM 확보량 및 가려진 오버헤드(Framework Tax) 추적</b>', 
        font=dict(size=22), x=0.5, y=0.92
    ),
    yaxis_title=dict(text='<b>확보된 VRAM 용량 (GiB)</b>', font=dict(size=15)),
    barmode='group',
    bargroupgap=0.15, 
    bargap=0.2,       
    height=750,       
    yaxis=dict(
        range=[0, final_y_max + 0.3], 
        showgrid=True, gridcolor='#EAEAEA', zeroline=True, zerolinecolor='#CCCCCC'
    ),
    xaxis=dict(tickfont=dict(size=14, weight='bold')),
    font=dict(family="Malgun Gothic, AppleGothic, NanumGothic, sans-serif"),
    plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=13)),
    margin=dict(t=120, b=50, l=50, r=50)
)

fig.show()

In [1]:
import os
import re
import plotly.graph_objects as go

# =====================================================================
# 1. 데이터 파싱 및 추출 함수
# =====================================================================
def parse_log_file(file_path):
    if not os.path.exists(file_path): return [0.0] * 5
    with open(file_path, 'r', encoding='utf-8') as f:
        log_text = f.read()
    try:
        base_vram = float(re.search(r'순수 모델 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_8bit = float(re.search(r'8-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_4bit = float(re.search(r'4-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_3bit = float(re.search(r'3-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
        vram_2bit = float(re.search(r'2-bit 적재 VRAM:\s*([\d.]+)\s*GB', log_text).group(1))
    except AttributeError:
        return [0.0] * 5
    return [base_vram, vram_8bit, vram_4bit, vram_3bit, vram_2bit]

# =====================================================================
# 2. 경로 설정 및 데이터 연산
# =====================================================================
log_paths = {
    'Llama_3.2_1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Llama_3.2_1B\master_log_Llama_3.2_1B_Distillation_20260406_031434.log',
    'Qwen2.5_1.5B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\Qwen2.5_1.5B\master_log_Qwen2.5_1.5B_Base_RLHF_20260406_010213.log',
    'TinyLlama_1.1B': r'C:\Users\user\SLM\02_cuda_aligned\all_logs\TinyLlama_1.1B\master_log_TinyLlama_1.1B_Base_Scratch_20260405_231336.log'
}

model_data = {}
for model_name, path in log_paths.items():
    model_data[model_name] = parse_log_file(path)

labels = ['원본 (BF16)', 'GPTQ 8-bit', 'GPTQ 4-bit', 'GPTQ 3-bit', 'GPTQ 2-bit']

colors = {'Llama_3.2_1B': '#BDE0FE', 'Qwen2.5_1.5B': '#E1C6E9', 'TinyLlama_1.1B': '#FFE0B2'}
line_colors = {'Llama_3.2_1B': '#5A9BD5', 'Qwen2.5_1.5B': '#9B59B6', 'TinyLlama_1.1B': '#E67E22'}

# =====================================================================
# 3. Plotly 단일 막대그래프 시각화 
# =====================================================================
fig = go.Figure()

for model_name, vram_values in model_data.items():
    text_labels = []
    for i, val in enumerate(vram_values):
        if val == 0:
            text_labels.append("")
        elif i == 0 or vram_values[i-1] == 0:
            text_labels.append(f"<b>{val:.2f}</b>")
        else:
            diff = vram_values[i-1] - val
            text_labels.append(f"<b>{val:.2f}</b><br><span style='font-size:11px; color:#555555;'>↓ {diff:.2f}</span>")

    fig.add_trace(go.Bar(
        name=model_name,
        x=labels,
        y=vram_values,
        text=text_labels,
        textposition='outside',
        textfont=dict(size=13, color='#444444'), # 폰트 크기 미세 축소
        marker_color=colors[model_name],
        marker_line_color=line_colors[model_name], 
        marker_line_width=1.5,
        hovertemplate="<b>%{x}</b><br>모델: " + model_name + "<br>VRAM: %{y:.2f} GiB<extra></extra>"
    ))

# # =====================================================================
# # 4. 골든 크로스(4-bit) 구간 시각적 하이라이트 적용
# # =====================================================================
# fig.add_vrect(
#     x0=1.5, x1=2.5,
#     fillcolor="#F0F4F8", opacity=0.6,
#     layer="below", line_width=2, line_dash="dash", line_color="#7F8C8D",
#     annotation_text="<b>골든 크로스 발생 구간</b><br><span style='font-size:11px'>Llama & Qwen 1.52 GiB 수렴</span>",
#     annotation_position="top left",
#     annotation_font_size=13, # 어노테이션 폰트 크기 축소
#     annotation_font_color="#D35400"
# )

# =====================================================================
# 5. 레이아웃 및 여백 설정 (높이 2/3 축소 반영)
# =====================================================================
max_vram = max([max(v) for v in model_data.values() if v]) if any(model_data.values()) else 3.5

fig.update_layout(
    # [수정됨] 전체 크기 축소에 맞춰 제목 폰트 사이즈 조정 (24 -> 20)
    title=dict(
        text='<b>4-bit 양자화의 골든 크로스 (가성비 역전 현상)</b><br><span style="font-size:13px; color:#555;">동일한 VRAM 제약 하에서 체급이 더 큰 모델(Qwen 1.5B)의 운용이 가능해지는 구조적 수렴점</span>', 
        font=dict(size=20), x=0.5, y=0.92
    ),
    yaxis_title=dict(text='<b>VRAM 적재량 (GiB)</b>', font=dict(size=13)),
    # ★ [핵심 수정] 750px -> 500px 로 높이 축소
    height=500, 
    barmode='group',
    bargroupgap=0.1,
    font=dict(family="Malgun Gothic, AppleGothic, NanumGothic, sans-serif"),
    plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(size=12)),
    # [수정됨] 높이가 줄어든 만큼 상하 여백(t, b)을 다이어트하여 막대 공간 확보
    margin=dict(t=100, b=50, l=60, r=40) 
)

fig.update_yaxes(range=[0, max_vram + 0.8], showgrid=True, gridcolor='#EAEAEA', zeroline=True, zerolinecolor='#CCCCCC')
fig.update_xaxes(tickfont=dict(size=13, weight='bold'))

fig.show()